## Transformer sentiment analysis

Here We're taking the code from [this tutorial](https://www.datacamp.com/tutorial/building-a-transformer-with-py-torch) and adapting it to do sentiment analysis on movie reviews.

**Well, we tried. Check out the other file in this folder instead.**

In [1]:
#Import all the things
import math
import copy
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import spacy
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torch.nn.utils.rnn import pad_sequence

from torch.utils.data import DataLoader

In [2]:
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

100%|██████████| 25.7M/25.7M [00:01<00:00, 20.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/versions/1


In [3]:
# We could probably do without Pandas
df = pd.read_csv(path + "/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
spacy.prefer_gpu() # Add this line to prefer GPU
nlp = spacy.load("en_core_web_sm")
import en_core_web_sm
nlp = en_core_web_sm.load()
doc = nlp("This is a sentence.")

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

Divide the data: 80% for training

In [6]:
# divide df into df_train and df_valid
df_train = df.sample(frac=0.8, random_state=0)
df_valid = df.drop(df_train.index)

In [27]:
len(df_train), len(df_valid)

(40000, 10000)

This uses an insane amount of RAM. Sorry.

In [7]:
if not Path('padded_training_sequences.pt').exists():
  print("Generating padded training sequences...
  training_embeddings = []
  training = df_train['review'].tolist()
  for i in range(len(training)): # Process the first 1000 for demonstration
      doc = nlp(training[i])
      # Get word embeddings for each token in the document and convert to numpy array
      word_embeddings = [w.vector.get() if hasattr(w.vector, 'get') else w.vector for w in doc] # .get() if it's a cupy array
      training_embeddings.append(word_embeddings)
      if i % 1000 == 0:
          print(f'Processed {i} reviews')

  # Convert the list of lists into a list of tensors, handling empty lists
  training_word_embedding_tensors = [torch.tensor(embeddings) for embeddings in training_embeddings if embeddings] # Only process non-empty embedding lists

  # Pad the sequences
  # `batch_first=True` means the batch dimension is the first dimension
  # `padding_value=0.0` means we will pad with zeros
  padded_sequences = pad_sequence(training_word_embedding_tensors, batch_first=True, padding_value=0.0)
  print("Shape of padded sequences:", padded_sequences.shape)
  # The shape will be [number_of_reviews, max_sequence_length, embedding_dimension]

Processed 0 reviews
Processed 1000 reviews
Processed 2000 reviews
Processed 3000 reviews
Processed 4000 reviews
Processed 5000 reviews
Processed 6000 reviews
Processed 7000 reviews
Processed 8000 reviews
Processed 9000 reviews
Processed 10000 reviews
Processed 11000 reviews
Processed 12000 reviews
Processed 13000 reviews
Processed 14000 reviews
Processed 15000 reviews
Processed 16000 reviews
Processed 17000 reviews
Processed 18000 reviews
Processed 19000 reviews
Processed 20000 reviews
Processed 21000 reviews
Processed 22000 reviews
Processed 23000 reviews
Processed 24000 reviews
Processed 25000 reviews
Processed 26000 reviews
Processed 27000 reviews
Processed 28000 reviews
Processed 29000 reviews
Processed 30000 reviews
Processed 31000 reviews
Processed 32000 reviews
Processed 33000 reviews
Processed 34000 reviews
Processed 35000 reviews
Processed 36000 reviews
Processed 37000 reviews
Processed 38000 reviews
Processed 39000 reviews


/tmp/ipython-input-4076384592.py:12: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  training_word_embedding_tensors = [torch.tensor(embeddings) for embeddings in training_embeddings if embeddings] # Only process non-empty embedding lists


Shape of padded sequences: torch.Size([40000, 2789, 96])


In [8]:
try:
  # Save the padded training sequences
  torch.save(padded_sequences, 'padded_training_sequences.pt')
except:
  print("Error saving padded sequences.")

In [38]:
#Path('padded_validation_sequences.pt').unlink()
if not Path('padded_validation_sequences.pt').exists():
  print("Generating padded validation sequences...")
  validation_embeddings = []
  validation = df_valid['review'].tolist()
  for i in range(len(validation)): # Process the first 1000 for demonstration
      doc = nlp(validation[i])
      # Get word embeddings for each token in the document and convert to numpy array
      word_embeddings = [w.vector.get() if hasattr(w.vector, 'get') else w.vector for w in doc] # .get() if it's a cupy array
      validation_embeddings.append(word_embeddings)
      if i % 1000 == 0:
          print(f'Processed {i} reviews')

  # Convert the list of lists into a list of tensors, handling empty lists
  validation_word_embedding_tensors = [torch.tensor(embeddings) for embeddings in validation_embeddings if embeddings] # Only process non-empty embedding lists

  # Pad the sequences
  # `batch_first=True` means the batch dimension is the first dimension
  # `padding_value=0.0` means we will pad with zeros
  padded_sequences = pad_sequence(validation_word_embedding_tensors, batch_first=True, padding_value=0.0)
  print("Shape of padded sequences:", padded_sequences.shape)
  # The shape will be [number_of_reviews, max_sequence_length, embedding_dimension]

In [39]:
# Save the padded validation sequences
try:
  torch.save(padded_sequences, 'padded_validation_sequences.pt')

  print("Padded sequences saved.")
  del(padded_sequences)
except Exception as e:
  print(f"Error saving padded sequences: {e}")

Error saving padded sequences: name 'padded_sequences' is not defined


In [50]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        # Ensure that the model dimension (d_model) is divisible by the number of heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        # Initialize dimensions
        self.d_model = d_model # Model's dimension
        self.num_heads = num_heads # Number of attention heads
        self.d_k = d_model // num_heads # Dimension of each head's key, query, and value

        # Linear layers for transforming inputs
        self.W_q = nn.Linear(d_model, d_model) # Query transformation
        self.W_k = nn.Linear(d_model, d_model) # Key transformation
        self.W_v = nn.Linear(d_model, d_model) # Value transformation
        self.W_o = nn.Linear(d_model, d_model) # Output transformation

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # Calculate attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # Apply mask if provided (useful for preventing attention to certain parts like padding)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)

        # Softmax is applied to obtain attention probabilities
        attn_probs = torch.softmax(attn_scores, dim=-1)

        # Multiply by values to obtain the final output
        output = torch.matmul(attn_probs, V)
        return output

    def split_heads(self, x):
        # Reshape the input to have num_heads for multi-head attention
        # Ensure x is on the correct device
        x = x.to(self.W_q.weight.device)
        # Use .shape instead of .size()
        batch_size, seq_length, d_model = x.shape
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        # Combine the multiple heads back to original shape
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, Q, K, V, mask=None):
        # Apply linear transformations and split heads
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        # Ensure mask is on the same device as Q, K, V if it exists
        if mask is not None:
            mask = mask.to(Q.device)

        # Perform scaled dot-product attention
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)

        # Combine heads and apply output transformation
        output = self.W_o(self.combine_heads(attn_output))
        return output

In [41]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        # Ensure input x is on the correct device before applying linear layers
        return self.fc2(self.relu(self.fc1(x.to(self.fc1.weight.device))))

In [42]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [43]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        # Ensure input x is on the correct device before passing to sub-modules
        x = x.to(self.norm1.weight.device)

        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

In [44]:
class Transformer(nn.Module):
    def __init__(self, d_model, num_heads, num_layers, d_ff, dropout):
        super(Transformer, self).__init__()
        # Removed embedding layers and positional encoding
        # self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        # self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        # self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        # self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]) # Decoder is not needed for classification

        self.fc = nn.Linear(d_model, 2) # Output size is 2 for binary sentiment classification
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, src, tgt):
        # Masks are not needed when using pre-computed embeddings and a single target
        src_mask = None
        tgt_mask = None
        return src_mask, tgt_mask

    def forward(self, src, tgt):
        # Directly use the input embeddings
        # Assuming input `src` is [batch_size, 1, d_model]
        src_mask = None # No mask needed for sequence length 1

        enc_output = src
        for enc_layer in self.encoder_layers:
            enc_output = enc_layer(enc_output, src_mask)

        # The output of the encoder is [batch_size, 1, d_model]
        # We need to flatten this to [batch_size, d_model] for the linear classifier
        output = self.fc(enc_output.squeeze(1)) # Squeeze to remove the sequence length dimension

        return output

In [45]:
d_model = 96
num_heads = 32
num_layers = 6
d_ff = 64
dropout = 0.1

transformer = Transformer(d_model, num_heads, num_layers, d_ff, dropout)

In [46]:
validation_embeddings = torch.load('padded_validation_sequences.pt')
training_embeddings = torch.load('padded_training_sequences.pt')

In [47]:
validation_embeddings.size(), training_embeddings.size()

(torch.Size([10000, 2537, 96]), torch.Size([40000, 2789, 96]))

Not sure why we're overfitting.

In [48]:
# Create DataLoader for validation set
x_valid = validation_embeddings
y_valid = torch.tensor(df_valid['sentiment'].map({'positive': 1, 'negative': 0}).values)
val_dataset = data.TensorDataset(x_valid, y_valid)
val_dataloader = DataLoader(val_dataset, batch_size=100, shuffle=True) # You can adjust the batch_size

In [51]:
# Create Dataset and DataLoader
x_train = training_embeddings
y_train = torch.tensor(df_train['sentiment'].map({'positive': 1, 'negative': 0}).values, dtype=torch.long) # Ensure target is Long type
train_dataset = data.TensorDataset(x_train, y_train)
train_dataloader = DataLoader(train_dataset, batch_size=2500, shuffle=True) # Increased batch size

criterion = nn.CrossEntropyLoss() # CrossEntropyLoss expects target of type Long
optimizer = optim.Adam(transformer.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9) # Reduced learning rate

# Check if CUDA is available and move model and criterion to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
transformer.to(device)
criterion.to(device)


print("Starting training...")
transformer.train()

for epoch in range(25):
    total_train_loss = 0
    for batch_src, batch_tgt in train_dataloader:
        # Move batch data to GPU and reshape source to [batch_size, 1, d_model]
        batch_src = batch_src.to(device).unsqueeze(1) # Add sequence length dimension
        # Target is already [batch_size] and is used as is by CrossEntropyLoss
        batch_tgt = batch_tgt.to(device)


        optimizer.zero_grad()
        output = transformer(batch_src, batch_tgt) # Pass batch_tgt, though it's not used in the redefined forward
        loss = criterion(output, batch_tgt) # CrossEntropyLoss expects output [batch_size, num_classes] and target [batch_size]
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_dataloader)

    # Evaluate on validation set
    transformer.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch_src, batch_tgt in val_dataloader:
            batch_src = batch_src.to(device).unsqueeze(1)
            batch_tgt = batch_tgt.to(device)
            val_output = transformer(batch_src, batch_tgt)
            val_loss = criterion(val_output, batch_tgt)
            total_val_loss += val_loss.item()
    avg_val_loss = total_val_loss / len(val_dataloader)

    print(f"Epoch: {epoch+1}, Training Loss: {avg_train_loss:.4f}, Validation Loss: {avg_val_loss:.4f}")
    transformer.train() # Set back to training mode

Using device: cuda
Starting training...


ValueError: too many values to unpack (expected 3)

In [ ]:
transformer.eval()

total_loss = 0
with torch.no_grad():
    for batch_src, batch_tgt in val_dataloader:
        # Move batch data to GPU and reshape source to [batch_size, 1, d_model]
        batch_src = batch_src.to(device).unsqueeze(1) # Add sequence length dimension
        # Target is already [batch_size]
        batch_tgt = batch_tgt.to(device)

        val_output = transformer(batch_src, batch_tgt) # Pass batch_tgt, though it's not used in the redefined forward
        loss = criterion(val_output, batch_tgt) # CrossEntropyLoss expects output [batch_size, num_classes] and target [batch_size]
        total_loss += loss.item()

avg_val_loss = total_loss / len(val_dataloader)
print(f"Validation Loss: {avg_val_loss}")

In [ ]:
# Calculate accuracy on validation set
transformer.eval() # Set the model to evaluation mode
correct_predictions = 0
total_predictions = 0

with torch.no_grad():
    for batch_src, batch_tgt in val_dataloader:
        batch_src = batch_src.to(device).unsqueeze(1)
        batch_tgt = batch_tgt.to(device)

        val_output = transformer(batch_src, batch_tgt)
        _, predicted = torch.max(val_output.data, 1)
        total_predictions += batch_tgt.size(0)
        correct_predictions += (predicted == batch_tgt).sum().item()

accuracy = correct_predictions / total_predictions
print(f"Validation Accuracy: {accuracy:.4f}")

In [ ]:
def run_model(inputdim, outputdim, hiddendim1, hiddenact1, hiddendim2, hiddenact2, hiddendim3=None, hiddenact3=None):
# Create Three Layered Neural Network
      print(f"First hidden layer: {hiddendim1} nodes, {hiddenact1.__name__} activation.")
      print(f"Second hidden layer: {hiddendim2} nodes, {hiddenact2.__name__} activation.")

      if hiddendim3 is not None:

            print(f"Third hidden layer: {hiddendim3} nodes, {hiddenact3.__name__} activation.")
            model = ThreeLayeredNN(inputdim, hiddendim1, hiddenact1,
                              hiddendim2, hiddenact2,
                              hiddendim3, hiddenact3, outputdim)
      else:
            model = TwoLayeredNN(inputdim, hiddendim1, hiddenact1,
                                    hiddendim2, hiddenact2, outputdim)

      # Cross Entropy Loss
      error = nn.CrossEntropyLoss()

      # SGD Optimizer
      learning_rate = 0.05
      momentum_rate = 0.9
      #optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
      optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=momentum_rate)

      # Let's train the model.
      print('Number of Epochs:',num_epochs)
      count = 0
      loss_list = []
      iteration_list = []
      accuracy_list = []
      for epoch in range(num_epochs):
        #print('Epoch Number',epoch)
        for i, (images, labels) in enumerate(train_loader):

            train = Variable(images.view(-1, 28*28))
            labels = Variable(labels)

            # Clear gradients
            optimizer.zero_grad()
            # Forward propagation
            outputs = model(train)
            # Calculate the Cross entropy loss (built in softmax)
            loss = error(outputs, labels)
            # Back propagation
            loss.backward()
            # Update parameters
            optimizer.step()

            #Print out accuracy and loss. (And store for later graphs)
            count += 1
            if count % 50 == 0:
                # Calculate Accuracy
                correct = 0
                total = 0
                # Predict test dataset
                for images, labels in test_loader:
                    test = Variable(images.view(-1, 28*28))

                    # Forward propagation
                    outputs = model(test)
                    # Get predictions from the maximum value
                    predicted = torch.max(outputs.data, 1)[1]
                    # Total number of labels
                    total += len(labels)
                    # Total correct predictions
                    correct += (predicted == labels).sum()

                accuracy = 100 * correct / float(total)

                # store loss and iteration
                loss_list.append(loss.data)
                iteration_list.append(count)
                accuracy_list.append(accuracy)

      print('Iteration: {}  Loss: {}  Accuracy: {} %'.format(count, loss.data, accuracy))
